### Uploading Parquet To Our Gcp Bucket 

In [ ]:
# Run This command 
gsutil -m cp -r data/pq/ gs://freeman-kestra-zoomcamp-bucket/parquet

# cp   -> linux copy command
# -r  -> This is recursive , meaning we want to go over each folder and get datas in them
# data/pq/      -> This is the location of our parquet file in our local storage
# gs://freeman-kestra-zoomcamp-bucket/parquet       -> This is our gcp bucke folder where we wamt our file to live in


### Connecting and using our data in gcp 

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [ ]:
credentials_location = '/home/freeman/.google/credentials/google_credentials.json'

spark = SparkSession.builder \
    .appName("test") \
    .master("local[*]") \
    .config("spark.jars", "./lib/gcs-connector-hadoop3-2.2.5.jar") \  
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location) \
    .getOrCreate()  # will reuse existing SparkContext if one exists

# spark = SparkSession.builder\
#         .master("local[*]")\
#         .appName("test")\
#         .getOrCreate()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [4]:

# Get the SparkContext
sc = spark.sparkContext

# Configure Hadoop to work with GCS
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

In [20]:
df_green = spark.read.parquet('gs://freeman-kestra-zoomcamp-bucket/parquet/pq/green/2020/01/part-00000-79b4679f-39c4-4c2e-9138-4cef47b04db1-c000.snappy.parquet')

In [ ]:
df_green.show()

In [52]:
df_green = spark.read \
    .option("recursiveFileLookup", "true")\
    .option("mergeSchema","true")\
    .parquet("gs://freeman-kestra-zoomcamp-bucket/parquet/pq/green/2020")

In [50]:
df_green.count() 

447770

## Trying To Run This like below will raise error (no recursive)

In [ ]:
df_green = spark.read.parquet("gs://freeman-kestra-zoomcamp-bucket/parquet/pq/green/2020")